# TFLN adiabatic coupler: accuracy and runtime benchmark

Companion to the [BeamZ example](tfln_adiabatic_coupler.ipynb), based on the [Tidy3D LN coupler](https://www.flexcompute.com/tidy3d/examples/notebooks/AdiabaticCouplerLN/).

**Status: benchmark prepared, not run.** Default execution creates a case plan and performs configuration checks only. It does not launch FDTD, solve modes, import Tidy3D, or submit cloud jobs. No timing, convergence, or accuracy results are supplied.

The shared [setup module](../tfln_coupler.py) fixes geometry, optical-axis orientation, material values, source and monitor planes, and normalization. This benchmarks **nondispersive diagonal anisotropy**.

## 1. Setup and explicit run controls

Use the same checkout and kernel as the example. Tidy3D is optional and needed only for a fresh reference run; existing `.npz` reference spectra can be compared without it. The Tidy3D branch requires an installed client, configured credentials, and paid cloud execution when explicitly enabled later.

In [1]:
from pathlib import Path
from dataclasses import asdict, replace
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

# Works from the repository root, beamz/, or examples/notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "examples/tfln_coupler.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the BeamZ checkout; retain examples/tfln_coupler.py.")
sys.path.insert(0, str(ROOT))

import beamz as bz
from examples.tfln_coupler import (
    REFERENCE_URL, CouplerConfig, build_simulation, load_spectrum,
    modal_spectrum, plane_centres, resource_estimate, save_spectrum,
    waveguide_vertices,
)

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"te": "#2463A0", "tm": "#B46B12"}

In [2]:
RUN_BEAMZ = False
RUN_SENSITIVITY = False  # Additional full runs; requires RUN_BEAMZ too.
RUN_TIDY3D = False       # Explicit opt-in to paid cloud runs on the simulation PC.
LOAD_SAVED = False       # Enable to analyze previously completed benchmark cases.
BACKEND = "jax"
BASE = CouplerConfig()
OUTPUT_DIR = ROOT / "data/tfln_adiabatic_coupler/benchmark"
REFERENCE_DIR = ROOT / "data/tfln_adiabatic_coupler/tidy3d"

MESH_CONFIGS = [replace(BASE, dx_um=dx) for dx in (0.08, 0.06, 0.05)]
SENSITIVITY_CONFIGS = {
    "longer_time": replace(BASE, run_time_s=60e-12),
    "larger_padding_and_PML": replace(BASE, domain_y_um=8.8, domain_z_um=6.0, pml_um=0.8),
}
# Provisional engineering targets in absolute power fraction, not established solver tolerances.
MESH_TARGET = 0.02
REFERENCE_TARGET = 0.03

## 2. Benchmark definition and case costs

Measure TE₀/TM₀ cross-port transfer, through-port power, and input reflection over 1.45–1.65 µm. Use the **same physical 2 mm device** at each mesh spacing; shortening the coupler would change its adiabaticity and is not an accuracy benchmark.

Mesh convergence uses 80, 60, and 50 nm spacing. The finest result is a numerical comparison baseline, not ground truth. The sensitivity cases compare 40 vs 60 ps and enlarged padding/PML at 60 nm. PML thickness is rounded up to grid cells and the domain is snapped outward by at most one cell.

The resource figure is the six-field storage lower bound only. Actual peak memory is not measured by this notebook. Uniform full-domain grids are intentionally expensive; decide whether the simulation PC has sufficient resources before enabling any full case.

In [3]:
case_plan = []
for config in MESH_CONFIGS:
    estimate = resource_estimate(config)
    case_plan.append({"dx_nm": config.dx_um * 1000,
                      "cells_million": estimate["cells"] / 1e6,
                      "fields_GiB_min": estimate["six_float32_fields_GiB_lower_bound"],
                      "steps_estimate": estimate["vacuum_CFL_steps_estimate"],
                      "polarizations": 2, "config_id": config.identity})
for row in case_plan:
    print(row)
assert len({c.identity for c in MESH_CONFIGS}) == 3
assert all(c.length_um == 2000.0 and c.run_time_s == BASE.run_time_s for c in MESH_CONFIGS)
for c in (*MESH_CONFIGS, *SENSITIVITY_CONFIGS.values()):
    np.testing.assert_allclose(c.frequencies, BASE.frequencies)
    assert np.all(np.diff(c.frequencies) > 0)
print("Case-plan checks passed. No solver executed.")

{'dx_nm': 80.0, 'cells_million': 164.125, 'fields_GiB_min': 3.66847962141037, 'steps_estimate': 262251, 'polarizations': 2, 'config_id': 'f7b8485da633'}
{'dx_nm': 60.0, 'cells_million': 392.489886, 'fields_GiB_min': 8.772832587361336, 'steps_estimate': 349668, 'polarizations': 2, 'config_id': 'a33368ff63c2'}
{'dx_nm': 50.0, 'cells_million': 672.256, 'fields_GiB_min': 15.026092529296875, 'steps_estimate': 419601, 'polarizations': 2, 'config_id': '91510cc8583c'}
Case-plan checks passed. No solver executed.


## 3. Run BeamZ and persist each completed case

Only completed runs produce files. `run_wall_s` includes specification construction, rasterization, compilation, execution, and result decoding; `analysis_wall_s` separately records modal projection. BeamZ's `performance.runtime_s` and derived GCUPS cover its timed executable, excluding setup/JIT. These are single-run observations, not repeated warm-performance statistics. Hardware, versions, commit, and dirty-worktree status are recorded.

The benchmark omits the xy field monitor to keep acquisition costs consistent across cases. It uses measured incident modal power for normalization and does not clip excess power. An explicit backend avoids silently comparing different execution paths. Runs are sequential; large results and compilation caches are released after each case.

In [4]:
def run_beamz_case(config, polarization):
    import gc
    import jax
    import platform
    import subprocess

    start = time.perf_counter()
    simulation = build_simulation(config, polarization, include_field=False)
    result = simulation.run(backend=BACKEND, progress=True, performance=True)
    run_wall_s = time.perf_counter() - start
    analysis_start = time.perf_counter()
    spectrum = modal_spectrum(result.mode, config)
    analysis_wall_s = time.perf_counter() - analysis_start
    if not np.all(spectrum["valid"]):
        print("Weak incident bins detected; inspect source spectrum before accepting this case.")
    if not all(np.all(np.isfinite(spectrum[k][spectrum["valid"]]))
               for k in ("T_cross", "T_through", "R_input")):
        raise RuntimeError("Non-finite modal results; benchmark case was not saved.")
    metadata = dict(
        backend=BACKEND, beamz_version=bz.__version__, jax_version=jax.__version__,
        python_version=platform.python_version(), devices=[str(d) for d in jax.devices()],
        git_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        worktree_dirty=bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True)),
        include_field=False, run_wall_s=run_wall_s, analysis_wall_s=analysis_wall_s,
        performance=asdict(result.performance) if result.performance else None,
    )
    path = save_spectrum(OUTPUT_DIR, "beamz", polarization, config, spectrum, **metadata)
    print("Saved", path)
    del simulation, result
    jax.clear_caches()
    gc.collect()
    return spectrum, metadata

beamz_results = {}
if RUN_BEAMZ:
    configs = MESH_CONFIGS + (list(SENSITIVITY_CONFIGS.values()) if RUN_SENSITIVITY else [])
    for config in configs:
        for polarization in ("te", "tm"):
            beamz_results[(config.identity, polarization)] = run_beamz_case(config, polarization)
else:
    print("BeamZ benchmark runs deferred.")

BeamZ benchmark runs deferred.


## 4. Optional matched Tidy3D reference

The helper creates the same centred waveguides, constant diagonal indices, uniform spacing, frequency bins, mode apertures, and input normalization as the BeamZ case. It uses the full domain without symmetry and disables early shutoff to retain the specified time window. Tidy3D places PML outside its `size`; the helper subtracts the absorbing-layer thickness from BeamZ's total domain to match the non-PML interior. Verify the realized grids on the simulation PC; subpixel treatments and PML implementations still differ.

This is a **matched adaptation**, not the original notebook's automatic mesh and symmetry settings. Inspect mode ordering in both solvers before treating their selected mode 0 as equivalent. Cloud turnaround is not comparable with local kernel runtime.

`RUN_TIDY3D=True` will upload, estimate cost, run, and save two cloud simulations at the finest mesh. Leave it false to use saved references only. No credentials or task IDs are bundled.

In [5]:
tidy_results = {}
reference_config = MESH_CONFIGS[-1]
if RUN_TIDY3D:
    import tidy3d as td
    import tidy3d.web as web
    from examples.tfln_coupler import build_tidy3d_simulation

    REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
    for polarization in ("te", "tm"):
        reference_sim = build_tidy3d_simulation(reference_config, polarization)
        name = f"tfln_{polarization}_{reference_config.identity}"
        reference_sim.to_file(str(REFERENCE_DIR / f"{name}_simulation.json"))
        job = web.Job(simulation=reference_sim, task_name=name)
        print("Estimated maximum FlexCredits:", job.estimate_cost())
        reference_data = job.run(path=str(REFERENCE_DIR / f"{name}.hdf5"))
        spectrum = modal_spectrum(lambda name: reference_data[name], reference_config)
        metadata = dict(tidy3d_version=td.__version__, task_id=job.task_id,
                        include_field=False, reference_kind="matched_uniform_grid")
        save_spectrum(REFERENCE_DIR, "tidy3d", polarization, reference_config, spectrum, **metadata)
        tidy_results[(reference_config.identity, polarization)] = (spectrum, metadata)
        del reference_data
else:
    print("Tidy3D reference runs disabled; no cloud actions performed.")

Tidy3D reference runs disabled; no cloud actions performed.


## 5. Load completed results on a later session

Files carry a configuration hash and embedded configuration/normalization metadata. Mismatches raise an error. Missing cases remain missing and are never filled with synthetic values or the published 90% target. Copy the `data/tfln_adiabatic_coupler/` directory along with the checkout to preserve runs across machines.

In [6]:
if LOAD_SAVED:
    for config in (*MESH_CONFIGS, *SENSITIVITY_CONFIGS.values()):
        for polarization in ("te", "tm"):
            key = (config.identity, polarization)
            if key in beamz_results:
                continue
            try:
                beamz_results[key] = load_spectrum(OUTPUT_DIR, "beamz", polarization, config)
            except FileNotFoundError:
                print("Missing BeamZ case:", config.identity, polarization)
    for polarization in ("te", "tm"):
        key = (reference_config.identity, polarization)
        if key not in tidy_results:
            try:
                tidy_results[key] = load_spectrum(REFERENCE_DIR, "tidy3d", polarization, reference_config)
            except FileNotFoundError:
                print("Missing Tidy3D reference:", polarization)

## 6. Spectra and mesh convergence

Compare the same polarization and frequency bins. Report maximum absolute spectral error and RMS error, in power fraction. A target is met only if every bin is valid and the measured maximum error is below that provisional target. The plots retain values outside [0, 1] so numerical problems remain visible.

In [7]:
def comparison(left, right):
    np.testing.assert_allclose(left["frequency_hz"], right["frequency_hz"], rtol=1e-12)
    valid = left["valid"] & right["valid"]
    valid &= np.isfinite(left["T_cross"]) & np.isfinite(right["T_cross"])
    if not np.any(valid):
        raise ValueError("No valid common frequency bins.")
    delta = left["T_cross"][valid] - right["T_cross"][valid]
    return {"max_abs_error": float(np.max(np.abs(delta))),
            "rms_error": float(np.sqrt(np.mean(delta**2))),
            "valid_bins": int(valid.sum()), "total_bins": int(valid.size)}

comparison_rows = []
if beamz_results:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, polarization in zip(axes, ("te", "tm")):
        finest = beamz_results.get((MESH_CONFIGS[-1].identity, polarization))
        for config, style in zip(MESH_CONFIGS, (":", "--", "-")):
            entry = beamz_results.get((config.identity, polarization))
            if entry is None:
                continue
            spectrum, _ = entry
            order = np.argsort(spectrum["wavelength_um"])
            ax.plot(spectrum["wavelength_um"][order], spectrum["T_cross"][order],
                    style, color=COLORS[polarization], label=f"BeamZ {config.dx_um * 1000:g} nm")
            if finest is not None and config != MESH_CONFIGS[-1]:
                metrics = comparison(spectrum, finest[0])
                comparison_rows.append(dict(kind="mesh_vs_finest", polarization=polarization,
                                            dx_nm=config.dx_um * 1000, **metrics,
                                            target_met=(metrics["max_abs_error"] <= MESH_TARGET
                                                        and metrics["valid_bins"] == metrics["total_bins"])))
        reference = tidy_results.get((reference_config.identity, polarization))
        if reference is not None:
            spectrum, _ = reference
            order = np.argsort(spectrum["wavelength_um"])
            ax.plot(spectrum["wavelength_um"][order], spectrum["T_cross"][order],
                    "-.", color="0.25", label="Tidy3D matched reference")
            if finest is not None:
                metrics = comparison(finest[0], spectrum)
                comparison_rows.append(dict(kind="beamz_vs_tidy3d", polarization=polarization,
                                            **metrics, target_met=(metrics["max_abs_error"] <= REFERENCE_TARGET
                                                                  and metrics["valid_bins"] == metrics["total_bins"])))
        ax.set(title=polarization.upper(), xlabel="Wavelength (µm)", ylabel="Cross-port modal transfer")
        if ax.lines:
            ax.legend(fontsize=9)
    plt.show()
    for row in comparison_rows:
        print(row)
else:
    print("No completed cases: convergence and reference agreement are not evaluated.")

No completed cases: convergence and reference agreement are not evaluated.


## 7. Duration/boundary sensitivity and runtime observations

A small mesh difference does not prove the time window or boundaries are adequate. Compare each sensitivity case with the 60 nm baseline. The sum below includes only three selected-mode channels; it is not a closed-surface energy balance. Negative or greater-than-unity values should be investigated, not clipped away.

No speedup is computed against Tidy3D cloud turnaround. Solver-reported runtime and end-to-end latency remain separate.

In [8]:
for label, config in SENSITIVITY_CONFIGS.items():
    for polarization in ("te", "tm"):
        baseline = beamz_results.get((BASE.identity, polarization))
        candidate = beamz_results.get((config.identity, polarization))
        if baseline is not None and candidate is not None:
            print(label, polarization, comparison(candidate[0], baseline[0]))
        else:
            print(label, polarization, "not evaluated")

runtime_rows = []
for (identity, polarization), (spectrum, metadata) in beamz_results.items():
    perf = metadata.get("performance")
    channels = spectrum["T_cross"] + spectrum["T_through"] + spectrum["R_input"]
    row = {"config_id": identity, "polarization": polarization,
           "run_wall_s": metadata.get("run_wall_s"),
           "analysis_wall_s": metadata.get("analysis_wall_s"),
           "max_selected_channel_sum": float(np.nanmax(channels)),
           "kernel_runtime_s": perf["runtime_s"] if perf else None,
           "kernel_GCUPS": (perf["cells"] * perf["steps"] / perf["runtime_s"] / 1e9) if perf else None}
    runtime_rows.append(row)
    print(row)
if not runtime_rows:
    print("No timing measurements yet.")

longer_time te not evaluated
longer_time tm not evaluated
larger_padding_and_PML te not evaluated
larger_padding_and_PML tm not evaluated
No timing measurements yet.


## Handoff and acceptance

On the simulation PC:

1. Verify available memory/backend and inspect TE₀/TM₀ profiles in the example notebook.
2. Run mesh cases sequentially, keeping the physical device and monitor definitions fixed.
3. Enable the longer-time and larger-boundary cases, then examine their changes.
4. Generate or load the matched Tidy3D reference and inspect all valid frequency bins. The published qualitative transfer target alone is insufficient for validation.
5. Save executed notebooks and the per-case `.npz`/reference files. Record remaining discrepancies rather than declaring success solely because the runs completed.

**Only setup sanity checks are intended on the current machine.** Numerical convergence, actual memory capacity, mode equivalence, timing repeatability, and cross-solver agreement remain unverified. No push is required for local notebook preparation.